# ML-09 — Validation and Research Claim Audit

## 1. Two Paper Findings + My Methodology Questions

Below we apply a constructive engineering review to two empirical findings from FlyRank's research paper on organic content performance and AI/search dynamics:

1. **Paper Finding 1 — Cross-Channel Visibility & Content Decay Patterns Across Multi-Month Search Windows:**
   * *Where does the label come from?* In aggregated 90-day snapshots (`fact_content_query_90d` / starter panel), decay labels (`trend_direction`, `trend_pct`) compare a recent sub-window against a prior baseline window. If static 90-day feature totals (`impressions_90d`, `clicks_90d`) span the exact same 90-day calendar range as the trend label, the features partially overlap the outcome window.
   * *Methodology Question:* How cleanly are pre-decision observation windows separated from forward outcome windows across clients with different onboarding dates (`gsc_data_start`), and do reported gains hold when evaluated strictly on held-out clients (`GroupShuffleSplit`) rather than pooled URL-level splits?

2. **Paper Finding 2 — Engagement, CTR, and Structural Missingness Across Content Types:**
   * *Where does the label / signal come from?* Search Console and GA4 telemetry exhibit structural missingness (`avg_position == 0` encodes *"no ranking data"* rather than rank #0, and pre-onboarding dates are zero-filled unless filtered by `gsc_data_available IS TRUE`). Additionally, rate metrics (`ctr` on a `0–100` scale, or cross-system ratios) follow heavy-tailed distributions.
   * *Methodology Question:* When comparing performance tiers across content categories, how much of the observed separation reflects true user behavior versus structural telemetry coverage (`has_position_data`, `client_has_ga4`), and are non-parametric / rank-based metrics (`PR-AUC`, `Precision@K`, medians) reported alongside parametric averages?

## 2. My Model Under an Honest Split (Before / After)

We re-run our Week-5 Random Forest classifier (`max_depth=6`, `random_state=42`) on the exact same pre-decision feature set under two validation designs:
* **Before (Naive Random Split — `train_test_split`):** Mixes URLs from the same client across train and test sets, allowing the model to memorize client-level traffic baselines.
* **After (Honest Client-Grouped Split — `GroupShuffleSplit` on `client_id`):** Holds out entire unseen clients (`20%` of client groups), measuring true cross-portfolio generalization alongside the positive base rate.

In [2]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

SEED = 42

# Localiza a raiz do repositório automaticamente
if os.path.exists("data/raw/content_refresh_anonymized.csv"):
    REPO_ROOT = "."
elif os.path.exists("../../data/raw/content_refresh_anonymized.csv"):
    REPO_ROOT = "../.."
else:
    raise FileNotFoundError(f"Dataset não encontrado a partir de {os.getcwd()}")

data_path = os.path.join(REPO_ROOT, "data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(data_path)

# Define o alvo binário observando os valores reais do CSV
if "is_declining" in df.columns:
    df["is_declining_label"] = df["is_declining"].astype(int)
elif "trend_direction" in df.columns:
    df["is_declining_label"] = (df["trend_direction"].astype(str).str.lower().isin(["declining", "down"])).astype(int)
elif "trend_pct" in df.columns:
    df["is_declining_label"] = (df["trend_pct"] <= -20.0).astype(int)
else:
    raise KeyError(f"Coluna de alvo não identificada: {list(df.columns)}")

assert df["is_declining_label"].nunique() == 2, "Erro: alvo degenerado."

# Engenharia de features pré-decisão alinhada com a Semana 5 (tratando sentinel avg_position == 0)
age_col = "days_since_last_update" if "days_since_last_update" in df.columns else "content_age_days"
df["has_position_data"] = (df["avg_position"] > 0).astype(int)
pos_median = df.loc[df["avg_position"] > 0, "avg_position"].median()
df["avg_position_clean"] = df["avg_position"].replace(0, pos_median)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"].clip(lower=0))
df["log_clicks_90d"] = np.log1p(df["clicks_90d"].clip(lower=0))
df["clicks_per_impression"] = df["clicks_90d"] / np.maximum(df["impressions_90d"], 1)

excluded_cols = {
    "content_id", "client_id", "url", "domain", "is_declining", "is_declining_label",
    "trend_direction", "trend_pct", "avg_position"
}
feature_cols = [
    c for c in df.select_dtypes(include=[np.number]).columns
    if c not in excluded_cols
]

X = df[feature_cols].fillna(0.0)
y = df["is_declining_label"].values

# 1. BEFORE: Naive Random Split (permite vazamento de padrões específicos de cada cliente)
X_tr_rand, X_te_rand, y_tr_rand, y_te_rand = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
rf_rand = RandomForestClassifier(n_estimators=150, max_depth=6, min_samples_leaf=20, random_state=SEED, n_jobs=-1)
rf_rand.fit(X_tr_rand, y_tr_rand)
probs_rand = rf_rand.predict_proba(X_te_rand)[:, 1]

pr_auc_rand = float(average_precision_score(y_te_rand, probs_rand))
roc_auc_rand = float(roc_auc_score(y_te_rand, probs_rand))

# 2. AFTER: Honest Client-Grouped Split (isola clientes inéditos no teste)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))

assert set(df.iloc[train_idx]["client_id"]).isdisjoint(set(df.iloc[test_idx]["client_id"]))

rf_group = RandomForestClassifier(n_estimators=150, max_depth=6, min_samples_leaf=20, random_state=SEED, n_jobs=-1)
rf_group.fit(X.iloc[train_idx], y[train_idx])
probs_group = rf_group.predict_proba(X.iloc[test_idx])[:, 1]

pr_auc_group = float(average_precision_score(y[test_idx], probs_group))
roc_auc_group = float(roc_auc_score(y[test_idx], probs_group))
base_rate_test = float(y[test_idx].mean())

split_comparison = pd.DataFrame([
    {
        "Validation Design": "Random Baseline (No Skill)",
        "Base Rate": round(base_rate_test, 4),
        "PR-AUC": round(base_rate_test, 4),
        "ROC-AUC": 0.5000,
    },
    {
        "Validation Design": "BEFORE: Naive Random Split (Same Clients Mixed)",
        "Base Rate": round(float(y_te_rand.mean()), 4),
        "PR-AUC": round(pr_auc_rand, 4),
        "ROC-AUC": round(roc_auc_rand, 4),
    },
    {
        "Validation Design": "AFTER: Honest Client-Grouped Split (Unseen Clients)",
        "Base Rate": round(base_rate_test, 4),
        "PR-AUC": round(pr_auc_group, 4),
        "ROC-AUC": round(roc_auc_group, 4),
    },
])

print("=== BEFORE vs. AFTER VALIDATION AUDIT ===")
print(split_comparison.to_string(index=False))
print(f"\nMeasured Memorization Inflation (Naive PR-AUC - Honest PR-AUC): {pr_auc_rand - pr_auc_group:+.4f}")

=== BEFORE vs. AFTER VALIDATION AUDIT ===
                                  Validation Design  Base Rate  PR-AUC  ROC-AUC
                         Random Baseline (No Skill)      0.511  0.5110   0.5000
    BEFORE: Naive Random Split (Same Clients Mixed)      0.542  0.8457   0.8304
AFTER: Honest Client-Grouped Split (Unseen Clients)      0.511  0.7532   0.7608

Measured Memorization Inflation (Naive PR-AUC - Honest PR-AUC): +0.0925


## 3. Leakage Audit & Real Error Examples

We run two concrete checks on the final Week-5 feature set under the honest client-grouped split:
1. **Adversarial Leakage Stress Test:** Inject a label-derived sibling column (`trend_pct` if available, or a noisy target proxy) to verify the exact magnitude of leakage inflation vs. our clean feature set, and programmatically assert zero overlap between `feature_cols` and prohibited columns (`trend_direction`, `trend_pct`, `is_declining_label`, `client_id`, `content_id`).
2. **Concrete Error Inspection:** Examine real held-out **False Positives** and **False Negatives** from the honest model to diagnose structural limitations of static 90-day snapshots.

In [3]:
# 1. Auditoria programática contra colunas proibidas
forbidden_cols = {"trend_direction", "trend_pct", "is_declining", "is_declining_label", "client_id", "content_id"}
leaked_in_features = forbidden_cols.intersection(set(feature_cols))
assert len(leaked_in_features) == 0, f"ERRO DE VAZAMENTO: {leaked_in_features}"

# Teste de estresse com coluna derivada do alvo (trend_pct se existir no CSV, ou proxy ruidoso do alvo)
X_corrupted = X.copy()
if "trend_pct" in df.columns:
    leak_col_name = "trend_pct"
    X_corrupted[leak_col_name] = df["trend_pct"].fillna(0.0)
else:
    leak_col_name = "leaked_target_proxy"
    rng = np.random.RandomState(SEED)
    X_corrupted[leak_col_name] = y + rng.normal(0, 0.15, size=len(y))

rf_leaky = RandomForestClassifier(n_estimators=150, max_depth=6, min_samples_leaf=20, random_state=SEED, n_jobs=-1)
rf_leaky.fit(X_corrupted.iloc[train_idx], y[train_idx])
probs_leaky = rf_leaky.predict_proba(X_corrupted.iloc[test_idx])[:, 1]
pr_auc_leaky = float(average_precision_score(y[test_idx], probs_leaky))

print("=== LEAKAGE STRESS TEST (ON HONEST CLIENT-GROUPED SPLIT) ===")
print(f"Clean Feature Set PR-AUC:                  {pr_auc_group:.4f}")
print(f"Corrupted Feature Set (+{leak_col_name}) PR-AUC: {pr_auc_leaky:.4f}  <-- Inflated by target leakage")
print("Programmatic exclusion assertion: PASSED (zero target/ID columns in feature_cols).\n")

# 2. Exemplos reais de erro no conjunto de teste agrupado por cliente
test_audit_df = df.iloc[test_idx].copy()
test_audit_df["pred_prob"] = probs_group

show_cols = ["content_id", "client_id", "impressions_90d", "avg_position", "ctr", age_col, "pred_prob"]

real_fps = test_audit_df[test_audit_df["is_declining_label"] == 0].sort_values("pred_prob", ascending=False).head(3)
real_fns = test_audit_df[test_audit_df["is_declining_label"] == 1].sort_values("pred_prob", ascending=True).head(3)

print("=== REAL ERROR EXAMPLES: TOP 3 FALSE POSITIVES (Predicted Decline, Observed Stable) ===")
print(real_fps[show_cols].to_string(index=False))

print("\n=== REAL ERROR EXAMPLES: TOP 3 FALSE NEGATIVES (Predicted Stable, Observed Decline) ===")
print(real_fns[show_cols].to_string(index=False))

# Guarda recibo JSON da auditoria da Semana 6
outputs_dir = os.path.join(REPO_ROOT, "work/outputs")
os.makedirs(outputs_dir, exist_ok=True)
w06_receipt = {
    "notebook": "w06_validation_audit.ipynb",
    "base_rate_test": round(base_rate_test, 4),
    "naive_random_pr_auc": round(pr_auc_rand, 4),
    "honest_grouped_pr_auc": round(pr_auc_group, 4),
    "memorization_gap_pr_auc": round(pr_auc_rand - pr_auc_group, 4),
    "leaky_stress_test_pr_auc": round(pr_auc_leaky, 4),
    "features_audited": feature_cols,
}
with open(os.path.join(outputs_dir, "w06_validation_audit_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(w06_receipt, f, indent=2)

=== LEAKAGE STRESS TEST (ON HONEST CLIENT-GROUPED SPLIT) ===
Clean Feature Set PR-AUC:                  0.7532
Corrupted Feature Set (+trend_pct) PR-AUC: 1.0000  <-- Inflated by target leakage
Programmatic exclusion assertion: PASSED (zero target/ID columns in feature_cols).

=== REAL ERROR EXAMPLES: TOP 3 FALSE POSITIVES (Predicted Decline, Observed Stable) ===
          content_id         client_id  impressions_90d  avg_position  ctr  days_since_last_update  pred_prob
content_09227e80fef5 client_8527a891e2               51           8.1  0.0                      92   0.766813
content_ef6e7d7cfe15 client_8527a891e2              264          22.2  0.0                     104   0.764970
content_c148e44db30d client_8527a891e2              335          31.3  0.0                     104   0.762415

=== REAL ERROR EXAMPLES: TOP 3 FALSE NEGATIVES (Predicted Stable, Observed Decline) ===
          content_id         client_id  impressions_90d  avg_position  ctr  days_since_last_update  pred_p

## 4. Claim Rewrite

Below we audit and rewrite overstated claims into objective, decision-support language grounded strictly in measured evidence:

* **Overstated Claim #1 (Before):** *"Our Random Forest model accurately predicts which pages will lose organic traffic and outperforms heuristic rules across all clients."*
  * **Rewritten Safe Claim (After):** *"On the anonymized panel evaluated under a client-grouped holdout split (`GroupShuffleSplit`), our regularized Random Forest model demonstrated a measurable directional improvement in ranking decaying pages (`PR-AUC` and `Precision@K` above the base rate and static rule baseline), making it suitable as a decision-support triage signal for editorial review queues."*

* **Overstated Claim #2 (Before):** *"Content age and search volume prove that old high-impression pages always decay unless refreshed."*
  * **Rewritten Safe Claim (After):** *"We observed that combining trailing demand (`impressions_90d`), valid SERP position (`avg_position > 0`), and update recency provides directional ranking power, though static 90-day snapshots still produce false positives on stable evergreen pages and require multi-period warehouse trajectories for stronger temporal separation."*

* **Scale & Telemetry Note:** Percentage rate columns in FlyRank datasets are stored on a `0–100` scale (e.g., `ctr = 0.76` means `0.76%`), and `avg_position == 0` encodes *"no ranking data"* rather than rank #0; both conventions are explicitly preserved in our preprocessing pipeline.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.